# Lesson 0a: Introduction to Deep Learning — Theory

This is the first notebook in the series. Before we touch a training loop or
a framework, we need to answer one question honestly: **why bother with deep
learning at all?** Why not just use a linear model — it's simpler, faster to
train, and easier to reason about.

This notebook answers that question from first principles, with a worked
example a linear model provably cannot solve, and then builds — by hand, in
NumPy, with no autograd — the smallest network that can solve it.

## Introduction

Imagine you run quality control on a factory line with two switches, A and
B. A warning light should turn on exactly when the switches **disagree** —
one on, one off. If both are off, or both are on, everything is fine.

You are handed a bin of past readings, each labeled "light on" or "light
off", and asked to build a classifier. You reach for the simplest tool you
own: a straight line (or, in more than two dimensions, a hyperplane) that
separates the "on" readings from the "off" readings. You start drawing lines.

You will not find one. Not because you haven't tried hard enough — no
straight line exists that solves this problem, for any choice of slope or
intercept. This is the XOR problem, and it is the smallest example that
demonstrates why linear models have a hard ceiling, and why *stacking*
non-linear layers — the core idea of deep learning — breaks through it.

By the end of this notebook you will have:
- proven that a linear model cannot fit XOR,
- learned the notation for a computational graph and a forward pass, and
- hand-built a two-layer network in NumPy that solves XOR, and plotted its
  decision boundary.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fixed seed: every random draw in this notebook is reproducible.
SEED = 0
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)

## Why Depth

### The data

XOR ("exclusive or") takes two binary inputs and returns 1 when exactly one
of them is 1:

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

This is our "two switches disagree" light from the introduction.

In [ ]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)

colors = np.where(y == 1, "crimson", "steelblue")
plt.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolor="k", zorder=3)
for (px, py), label in zip(X, y):
    plt.annotate(f"y={int(label)}", (px, py), textcoords="offset points",
                 xytext=(12, 8))
plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title("XOR: crimson = 1, steelblue = 0")
plt.grid(alpha=0.3)
plt.show()

### Proof: no line separates this data

A linear classifier predicts $\hat{y} = 1$ when $w_1 x_1 + w_2 x_2 + b > 0$,
and $\hat{y} = 0$ otherwise. For it to fit XOR perfectly, all four of these
must hold simultaneously:

$$
\begin{aligned}
(0,0) \to 0: \quad & b < 0 \\
(0,1) \to 1: \quad & w_2 + b > 0 \\
(1,0) \to 1: \quad & w_1 + b > 0 \\
(1,1) \to 0: \quad & w_1 + w_2 + b < 0
\end{aligned}
$$

Add the second and third inequalities: $w_1 + w_2 + 2b > 0$, i.e.
$w_1 + w_2 > -2b$. But the first inequality gives $-2b > 0$, so
$w_1 + w_2 + b > -b > 0$ combined with the fourth inequality
($w_1 + w_2 + b < 0$) is a direct contradiction: we would need
$w_1 + w_2 + b$ to be **both** negative (constraint 4) and, from summing
constraints 2 and 3 minus constraint 1, positive. No choice of
$(w_1, w_2, b)$ satisfies all four constraints — the contradiction holds for
*every* real-valued line, not just the ones we happen to try.

Let's confirm this empirically too: search a wide grid of lines and confirm
none reaches 100% accuracy.

In [ ]:
def linear_predict(w1, w2, b, X):
    return (w1 * X[:, 0] + w2 * X[:, 1] + b > 0).astype(float)

rng = np.random.default_rng(SEED)
trials = rng.uniform(-5, 5, size=(20000, 3))  # (w1, w2, b)
best_acc, best_params = 0.0, None
for w1, w2, b in trials:
    acc = (linear_predict(w1, w2, b, X) == y).mean()
    if acc > best_acc:
        best_acc, best_params = acc, (w1, w2, b)

print(f"best accuracy found over 20000 random lines: {best_acc:.2f}")
print(f"best (w1, w2, b): {tuple(round(p, 3) for p in best_params)}")
assert best_acc < 1.0, "a perfect linear separator should not exist for XOR"


In [ ]:
# Plot the best line we found: it always misclassifies at least one point.
w1, w2, b = best_params
xx = np.linspace(-0.5, 1.5, 200)
plt.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolor="k", zorder=3)
if abs(w2) > 1e-9:
    yy = -(w1 * xx + b) / w2
    plt.plot(xx, yy, "k--", label="best line found")
plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title(f"Best achievable linear boundary ({best_acc:.0%} accuracy)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

No matter how the line is placed, at least one point ends up on the
wrong side — visually, XOR's two classes sit on opposite diagonals of the
unit square, and a single straight cut can never isolate diagonal corners
from each other.

This is the ceiling of linear models: they can only carve space with a
single hyperplane. **Depth** — composing multiple layers with a non-linear
activation between them — lets a network carve space with several
hyperplanes and *combine* the results, which is exactly enough to solve
XOR. That combination is what the rest of this notebook builds.

## The Computational Graph

Every network in this curriculum is a **computational graph**: a directed
sequence of tensor operations from input to output. We'll use consistent
notation throughout:

- $x \in \mathbb{R}^{d}$ — the input vector ($d$ = number of input features)
- $W^{(l)}, b^{(l)}$ — the weight matrix and bias vector of layer $l$
- $z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}$ — the pre-activation ("logit") of layer $l$
- $\phi$ — a non-linear activation function (e.g. sigmoid, ReLU, tanh)
- $a^{(l)} = \phi(z^{(l)})$ — the activation ("hidden state") of layer $l$, with $a^{(0)} = x$

A **forward pass** evaluates the graph layer by layer, left to right:

$$
x \;\xrightarrow{W^{(1)}, b^{(1)}}\; z^{(1)} \;\xrightarrow{\phi}\; a^{(1)}
\;\xrightarrow{W^{(2)}, b^{(2)}}\; z^{(2)} \;\xrightarrow{\phi}\; \hat{y}
$$

For a **two-layer network** (one hidden layer, one output layer) this is
just two matrix multiplies and two activations:

$$
a^{(1)} = \phi\big(W^{(1)} x + b^{(1)}\big), \qquad
\hat{y} = \phi\big(W^{(2)} a^{(1)} + b^{(2)}\big)
$$

Nothing here is learned yet — this section only fixes the *shape* and the
*notation* of the computation. Training (adjusting $W$ and $b$ by gradient
descent) is the subject of the companion practical notebook. In this theory
notebook we derive weights by hand to show that the *right shape* of network
is sufficient, independent of how those weights are found.

In [ ]:
# A tiny picture of the graph described above: two inputs feeding a
# hidden layer of two units, feeding a single output unit.
fig, ax = plt.subplots(figsize=(5, 4))
layer_x = {"input": 0, "hidden": 1, "output": 2}
positions = {
    "x1": (layer_x["input"], 1), "x2": (layer_x["input"], 0),
    "h1": (layer_x["hidden"], 1), "h2": (layer_x["hidden"], 0),
    "y": (layer_x["output"], 0.5),
}
edges = [("x1", "h1"), ("x1", "h2"), ("x2", "h1"), ("x2", "h2"),
         ("h1", "y"), ("h2", "y")]
for a, b in edges:
    ax.plot([positions[a][0], positions[b][0]],
            [positions[a][1], positions[b][1]], "k-", lw=1, zorder=1)
for name, (px, py) in positions.items():
    ax.scatter([px], [py], s=900, zorder=2,
               color="steelblue" if name.startswith("x") else
                     ("crimson" if name == "y" else "seagreen"))
    ax.annotate(name, (px, py), ha="center", va="center", color="white",
                fontsize=11, fontweight="bold", zorder=3)
ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 1.5)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["input $x$", "hidden $a^{(1)}$", "output $\\hat{y}$"])
ax.set_yticks([])
ax.set_title("Computational graph: a two-layer network")
for spine in ax.spines.values():
    spine.set_visible(False)
plt.show()

## A Two-Layer Network from Scratch

We now build the network sketched above, in raw NumPy, and hand-derive
weights that solve XOR. The trick — a classic in the deep learning theory
literature — is to make each hidden unit a simple linear boundary, and let
the output layer **combine** two such boundaries into a shape a single line
could never produce.

Concretely:
- hidden unit 1 approximates the OR gate: fires when $x_1 + x_2 \geq 1$
- hidden unit 2 approximates the NAND gate: fires when $x_1 + x_2 < 2$ (i.e. NOT AND)
- the output fires only when **both** hidden units fire — i.e. OR AND NAND,
  which is exactly XOR ($1,1 \to$ AND is true so NAND is false so output is 0; $0,0 \to$ OR is false so output is 0; $0,1$ and $1,0 \to$ both fire so output is 1)

We use a steep sigmoid as a smooth stand-in for a hard step function, so the
forward pass is differentiable in principle (useful later) while behaving
like a step function in practice.

In [ ]:
def sigmoid(z, steepness=12.0):
    """A steep sigmoid: approximates a hard step while staying smooth."""
    return 1.0 / (1.0 + np.exp(-steepness * z))

# Hand-derived weights: no training, no gradient descent. Layer 1 builds an
# OR detector and a NAND detector, each with margin so the steep sigmoid
# saturates cleanly (rather than sitting exactly on the 0.5 boundary);
# layer 2 ANDs them together.
W1 = np.array([[2.0, -2.0],    # -> hidden unit 1: OR   (fires when x1+x2 >= 1, margin 1)
               [2.0, -2.0]])   # -> hidden unit 2: NAND (fires unless x1+x2 == 2, margin 1)
b1 = np.array([-1.0, 3.0])

W2 = np.array([[1.0], [1.0]])  # AND the two hidden units together
b2 = np.array([-1.5])


def forward(X, W1, b1, W2, b2):
    """Forward-only two-layer network: no backward pass, this is inference.

    z1, a1  : pre-activation and activation of the hidden layer
    z2, yhat: pre-activation and activation (prediction) of the output layer
    """
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    yhat = sigmoid(z2)
    return yhat.ravel()


preds = forward(X, W1, b1, W2, b2)
for row, target, pred in zip(X, y, preds):
    print(f"x={row}, target={int(target)}, network output={pred:.3f}, "
          f"rounded={round(pred)}")

assert np.array_equal(np.round(preds), y), "the hand-built network must match XOR exactly"
print("\nAll four XOR cases match.")

### The decision boundary

The forward pass above only checked the four corners of the unit square.
The real payoff of depth is visible once we evaluate the network on a dense
grid over the whole plane: the two hidden units each contribute a straight
boundary, and their combination bends those two lines into a region no
single line could carve out.

In [ ]:
grid_n = 200
gx = np.linspace(-0.5, 1.5, grid_n)
gy = np.linspace(-0.5, 1.5, grid_n)
GX, GY = np.meshgrid(gx, gy)
grid_points = np.column_stack([GX.ravel(), GY.ravel()])

grid_preds = forward(grid_points, W1, b1, W2, b2).reshape(GX.shape)

plt.contourf(GX, GY, grid_preds, levels=20, cmap="RdBu_r", alpha=0.75)
plt.colorbar(label="network output $\\hat{y}$")
plt.contour(GX, GY, grid_preds, levels=[0.5], colors="k", linewidths=2)
plt.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolor="k", zorder=3)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title("Two-layer network decision boundary (solves XOR)")
plt.show()

The black contour at $\hat{y}=0.5$ is the network's decision
boundary. It is not a single straight line — it is a region bounded by two
lines, one contributed by each hidden unit, joined by the output layer. That
bend is precisely what a linear model cannot produce, and it is enough to
separate XOR's diagonal classes perfectly.

This is the entire idea of "depth" in miniature: each layer contributes
simple, linear pieces; composing them through a non-linearity lets the
network represent shapes no single layer could.

## Key Takeaways

- A linear model draws exactly one hyperplane through the input space. XOR's
  positive and negative examples sit on opposite diagonals of the unit
  square, and no single hyperplane separates them — we proved this both
  algebraically (a direct contradiction between the four required
  inequalities) and empirically (20,000 random lines, best case 75%
  accuracy).
- A **computational graph** is a directed sequence of tensor operations; a
  **forward pass** evaluates it layer by layer using
  $a^{(l)} = \phi(W^{(l)} a^{(l-1)} + b^{(l)})$. This notation is used
  throughout the rest of the curriculum.
- A two-layer network — one hidden layer plus an output layer, joined by a
  non-linear activation — can solve XOR with hand-derived weights, no
  training required. Each hidden unit contributes one linear boundary; the
  output layer combines them into a shape a single line cannot express.
- This is the core argument for depth: stacking non-linear layers buys
  **representational capacity** that no amount of scaling a linear model
  can recover. The companion practical notebook (0b) takes this same shape
  and trains its weights on real data instead of deriving them by hand.